# Notebook 03 — Chunking Experiments

## Goal
This notebook compares chunking strategies for multi-document cybersecurity RAG.

## Main tasks
- Reload cleaned documents
- Apply multiple chunking strategies
- Compare chunk counts and quality
- Verify metadata propagation
- Recommend the best chunking setup for production

In [1]:
# Standard library
from pathlib import Path
from typing import List, Dict, Any
import re
import json
from copy import deepcopy

# Data handling
import pandas as pd

# LangChain loaders
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import TextLoader

# LangChain splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Imports loaded successfully.")

Imports loaded successfully.


In [2]:
# ============================================================
# STEP 1 — Define paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "chunked_docs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_ROOT: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag
DATA_DIR: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\raw
OUTPUT_DIR: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\chunked_docs


In [3]:
# ============================================================
# STEP 2 — Detect file type
# ============================================================

def detect_file_type(file_path: Path) -> str:
    suffix = file_path.suffix.lower()

    if suffix == ".pdf":
        return "pdf"
    elif suffix == ".docx":
        return "docx"
    elif suffix == ".txt":
        return "txt"
    else:
        return "unsupported"

In [4]:
# ============================================================
# STEP 3 — Load one document
# ============================================================

def load_single_document(file_path: Path):
    file_type = detect_file_type(file_path)

    if file_type == "pdf":
        loader = PyPDFLoader(str(file_path))
        docs = loader.load()

    elif file_type == "docx":
        loader = Docx2txtLoader(str(file_path))
        docs = loader.load()

    elif file_type == "txt":
        loader = TextLoader(str(file_path), encoding="utf-8")
        docs = loader.load()

    else:
        raise ValueError(f"Unsupported file type: {file_path.suffix}")

    return docs

In [5]:
# ============================================================
# STEP 4 — Text cleaning helpers
# ============================================================

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\t", " ")
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = text.strip()

    return text


def remove_light_page_noise(text: str) -> str:
    if not isinstance(text, str):
        return ""

    lines = text.split("\n")
    cleaned_lines = [line.strip() for line in lines]
    text = "\n".join(cleaned_lines)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

In [6]:
# ============================================================
# STEP 5 — Standard metadata schema
# ============================================================

def build_standard_metadata(
    original_metadata: Dict[str, Any],
    global_doc_index: int
) -> Dict[str, Any]:
    source_path = str(original_metadata.get("source", "")).strip()
    source_obj = Path(source_path) if source_path else None

    file_name = source_obj.name if source_obj else "unknown"
    file_extension = source_obj.suffix.lower() if source_obj else ""
    doc_type = file_extension.replace(".", "") if file_extension else "unknown"

    page_number = original_metadata.get("page", None)

    standardized = {
        "document_id": f"doc_{global_doc_index:05d}",
        "file_name": file_name,
        "source_path": source_path,
        "doc_type": doc_type,
        "page_number": page_number,
        "section_title": None,
        "chunk_id": None,
        "loader_name": (
            "PyPDFLoader" if doc_type == "pdf"
            else "Docx2txtLoader" if doc_type == "docx"
            else "TextLoader" if doc_type == "txt"
            else "unknown"
        ),
        "original_metadata": dict(original_metadata)
    }

    return standardized

In [7]:
# ============================================================
# STEP 6 — Reload and clean documents
# ============================================================

all_files = sorted([p for p in DATA_DIR.iterdir() if p.is_file()])

raw_documents = []

for file_path in all_files:
    file_type = detect_file_type(file_path)

    if file_type == "unsupported":
        continue

    docs = load_single_document(file_path)
    raw_documents.extend(docs)

cleaned_documents = []

for idx, doc in enumerate(raw_documents):
    cleaned_text = clean_text(doc.page_content)
    cleaned_text = remove_light_page_noise(cleaned_text)

    doc.page_content = cleaned_text
    doc.metadata = build_standard_metadata(doc.metadata, idx)

    cleaned_documents.append(doc)

print("Total cleaned documents ready for chunking:", len(cleaned_documents))

Total cleaned documents ready for chunking: 114


In [8]:
# ============================================================
# STEP 7 — Preview documents before chunking
# ============================================================

preview_rows = []

for doc in cleaned_documents[:15]:
    preview_rows.append({
        "document_id": doc.metadata.get("document_id"),
        "file_name": doc.metadata.get("file_name"),
        "doc_type": doc.metadata.get("doc_type"),
        "page_number": doc.metadata.get("page_number"),
        "char_count": len(doc.page_content),
        "word_count": len(doc.page_content.split()),
        "text_preview": doc.page_content[:200].replace("\n", " ")
    })

preview_df = pd.DataFrame(preview_rows)
display(preview_df)

,document_id,file_name,doc_type,page_number,char_count,word_count,text_preview
0,doc_00000,audit.txt,txt,NaN,20154,3105,AUDIT TRAILS Audit trails maintain a record o...
1,doc_00001,cui-ssp-template-final.docx,docx,NaN,32742,4039,<<Insert name>> SYSTEM SECURITY PLAN Last Upda...
2,doc_00002,NIST.CSWP.29.pdf,pdf,0.0,195,24,National Institute of Standards and Technology...
3,doc_00003,NIST.CSWP.29.pdf,pdf,1.0,2251,316,T he NIST Cybersecurity Framework (CSF) 2.0 i ...
4,doc_00004,NIST.CSWP.29.pdf,pdf,2.0,502,82,NIST CSWP 29 The NIST Cybersecurity Framework ...
5,doc_00005,NIST.CSWP.29.pdf,pdf,3.0,1987,131,NIST CSWP 29 The NIST Cybersecurity Framework ...
6,doc_00006,NIST.CSWP.29.pdf,pdf,4.0,3362,504,NIST CSWP 29 The NIST Cybersecurity Framework ...
7,doc_00007,NIST.CSWP.29.pdf,pdf,5.0,2747,401,NIST CSWP 29 The NIST Cybersecurity Framework ...
8,doc_00008,NIST.CSWP.29.pdf,pdf,6.0,2312,340,NIST CSWP 29 The NIST Cybersecurity Framework ...
9,doc_00009,NIST.CSWP.29.pdf,pdf,7.0,2071,295,NIST CSWP 29 The NIST Cybersecurity Framework ...


In [9]:
# ============================================================
# STEP 8 — Define chunking strategies
# ============================================================

chunking_strategies = {
    "small_recursive": {
        "chunk_size": 500,
        "chunk_overlap": 80,
        "separators": ["\n\n", "\n", ". ", " ", ""]
    },
    "medium_recursive": {
        "chunk_size": 800,
        "chunk_overlap": 120,
        "separators": ["\n\n", "\n", ". ", " ", ""]
    },
    "large_recursive": {
        "chunk_size": 1200,
        "chunk_overlap": 150,
        "separators": ["\n\n", "\n", ". ", " ", ""]
    }
}

chunking_strategies

{'small_recursive': {'chunk_size': 500,
  'chunk_overlap': 80,
  'separators': ['\n\n', '\n', '. ', ' ', '']},
 'medium_recursive': {'chunk_size': 800,
  'chunk_overlap': 120,
  'separators': ['\n\n', '\n', '. ', ' ', '']},
 'large_recursive': {'chunk_size': 1200,
  'chunk_overlap': 150,
  'separators': ['\n\n', '\n', '. ', ' ', '']}}

In [10]:
# ============================================================
# STEP 9 — Apply chunking strategy
# ============================================================

def chunk_documents(documents, strategy_name: str, strategy_config: Dict[str, Any]):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=strategy_config["chunk_size"],
        chunk_overlap=strategy_config["chunk_overlap"],
        separators=strategy_config["separators"]
    )

    chunked_docs = []

    for doc in documents:
        split_docs = splitter.split_documents([doc])

        for chunk_index, chunk_doc in enumerate(split_docs):
            chunk_doc.metadata = deepcopy(chunk_doc.metadata)

            chunk_doc.metadata["parent_document_id"] = doc.metadata.get("document_id")
            chunk_doc.metadata["chunk_id"] = (
                f"{doc.metadata.get('document_id')}_chunk_{chunk_index:03d}"
            )
            chunk_doc.metadata["chunking_strategy"] = strategy_name
            chunk_doc.metadata["chunk_size"] = strategy_config["chunk_size"]
            chunk_doc.metadata["chunk_overlap"] = strategy_config["chunk_overlap"]

            chunked_docs.append(chunk_doc)

    return chunked_docs

In [11]:
# ============================================================
# STEP 10 — Generate chunks for all strategies
# ============================================================

all_chunk_results = {}

for strategy_name, strategy_config in chunking_strategies.items():
    chunked_docs = chunk_documents(cleaned_documents, strategy_name, strategy_config)
    all_chunk_results[strategy_name] = chunked_docs

    print(
        f"Strategy: {strategy_name} | "
        f"chunk_size={strategy_config['chunk_size']} | "
        f"overlap={strategy_config['chunk_overlap']} | "
        f"chunks_created={len(chunked_docs)}"
    )

Strategy: small_recursive | chunk_size=500 | overlap=80 | chunks_created=897
Strategy: medium_recursive | chunk_size=800 | overlap=120 | chunks_created=613
Strategy: large_recursive | chunk_size=1200 | overlap=150 | chunks_created=414


In [12]:
# ============================================================
# STEP 11 — Compare chunk counts across strategies
# ============================================================

comparison_rows = []

for strategy_name, chunked_docs in all_chunk_results.items():
    total_chars = sum(len(doc.page_content) for doc in chunked_docs)
    total_words = sum(len(doc.page_content.split()) for doc in chunked_docs)

    comparison_rows.append({
        "strategy": strategy_name,
        "num_chunks": len(chunked_docs),
        "avg_chars_per_chunk": total_chars / len(chunked_docs) if chunked_docs else 0,
        "avg_words_per_chunk": total_words / len(chunked_docs) if chunked_docs else 0
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

,strategy,num_chunks,avg_chars_per_chunk,avg_words_per_chunk
0,small_recursive,897,399.542921,55.899666
1,medium_recursive,613,621.047308,86.849918
2,large_recursive,414,891.915459,124.550725


In [13]:
# ============================================================
# STEP 12 — Per-file chunk statistics
# ============================================================

strategy_file_rows = []

for strategy_name, chunked_docs in all_chunk_results.items():
    for doc in chunked_docs:
        strategy_file_rows.append({
            "strategy": strategy_name,
            "file_name": doc.metadata.get("file_name"),
            "doc_type": doc.metadata.get("doc_type"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "char_count": len(doc.page_content),
            "word_count": len(doc.page_content.split())
        })

strategy_file_df = pd.DataFrame(strategy_file_rows)

grouped_strategy_file_df = (
    strategy_file_df.groupby(["strategy", "file_name", "doc_type"], dropna=False)
    .agg(
        num_chunks=("chunk_id", "count"),
        avg_chars=("char_count", "mean"),
        avg_words=("word_count", "mean")
    )
    .reset_index()
)

display(grouped_strategy_file_df)

,strategy,file_name,doc_type,num_chunks,avg_chars,avg_words
0,large_recursive,NIST.CSWP.29.pdf,pdf,90,798.711111,110.833333
1,large_recursive,audit.txt,txt,22,925.000000,142.772727
2,large_recursive,cui-ssp-template-final.docx,docx,33,1091.090909,134.727273
3,large_recursive,nist.sp.800-61r2.pdf,pdf,269,895.959108,126.401487
4,medium_recursive,NIST.CSWP.29.pdf,pdf,131,569.847328,78.854962
5,medium_recursive,audit.txt,txt,38,540.421053,83.578947
6,medium_recursive,cui-ssp-template-final.docx,docx,50,730.140000,90.060000
7,medium_recursive,nist.sp.800-61r2.pdf,pdf,394,632.002538,89.416244
8,small_recursive,NIST.CSWP.29.pdf,pdf,186,381.381720,52.951613
9,small_recursive,audit.txt,txt,58,368.017241,56.931034


In [14]:
# ============================================================
# STEP 13 — Inspect sample chunks from each strategy
# ============================================================

for strategy_name, chunked_docs in all_chunk_results.items():
    print("=" * 120)
    print(f"STRATEGY: {strategy_name}")
    print("=" * 120)

    for sample_doc in chunked_docs[:3]:
        print("\nCHUNK ID:", sample_doc.metadata.get("chunk_id"))
        print("FILE:", sample_doc.metadata.get("file_name"))
        print("PAGE:", sample_doc.metadata.get("page_number"))
        print("CHARS:", len(sample_doc.page_content))
        print("\nTEXT PREVIEW:\n")
        print(sample_doc.page_content[:1200])
        print("\n" + "-" * 120)

STRATEGY: small_recursive

CHUNK ID: doc_00000_chunk_000
FILE: audit.txt
PAGE: None
CHARS: 12

TEXT PREVIEW:

AUDIT TRAILS

------------------------------------------------------------------------------------------------------------------------

CHUNK ID: doc_00000_chunk_001
FILE: audit.txt
PAGE: None
CHARS: 487

TEXT PREVIEW:

Audit trails maintain a record of system activity both by system and
application processes and by user activity of systems and applications. In
conjunction with appropriate tools and procedures, audit trails can assist
in detecting security violations, performance problems, and flaws in
applications. This bulletin focuses on audit trails as a technical control
and discusses the benefits and objectives of audit trails, the types of
audit trails, and some common implementation issues.

------------------------------------------------------------------------------------------------------------------------

CHUNK ID: doc_00000_chunk_002
FILE: audit.txt
PAGE: None
CH

In [15]:
# ============================================================
# STEP 14 — Metadata propagation quality check
# ============================================================

metadata_rows = []

for strategy_name, chunked_docs in all_chunk_results.items():
    for doc in chunked_docs[:20]:
        metadata_rows.append({
            "strategy": strategy_name,
            "chunk_id": doc.metadata.get("chunk_id"),
            "parent_document_id": doc.metadata.get("parent_document_id"),
            "file_name": doc.metadata.get("file_name"),
            "doc_type": doc.metadata.get("doc_type"),
            "page_number": doc.metadata.get("page_number"),
            "source_path": doc.metadata.get("source_path"),
            "chunking_strategy": doc.metadata.get("chunking_strategy")
        })

metadata_df = pd.DataFrame(metadata_rows)
display(metadata_df)

,strategy,chunk_id,parent_document_id,file_name,doc_type,page_number,source_path,chunking_strategy
0,small_recursive,doc_00000_chunk_000,doc_00000,audit.txt,txt,None,d:\Abhishek\Project\LangChain\cybersecurity-mu...,small_recursive
1,small_recursive,doc_00000_chunk_001,doc_00000,audit.txt,txt,None,d:\Abhishek\Project\LangChain\cybersecurity-mu...,small_recursive
2,small_recursive,doc_00000_chunk_002,doc_00000,audit.txt,txt,None,d:\Abhishek\Project\LangChain\cybersecurity-mu...,small_recursive
3,small_recursive,doc_00000_chunk_003,doc_00000,audit.txt,txt,None,d:\Abhishek\Project\LangChain\cybersecurity-mu...,small_recursive
4,small_recursive,doc_00000_chunk_004,doc_00000,audit.txt,txt,None,d:\Abhishek\Project\LangChain\cybersecurity-mu...,small_recursive
5,small_recursive,doc_00000_chunk_005,doc_00000,audit.txt,txt,None,d:\Abhishek\Project\LangChain\cybersecurity-mu...,small_recursive
6,small_recursive,doc_00000_chunk_006,doc_00000,audit.txt,txt,None,d:\Abhishek\Project\LangChain\cybersecurity-mu...,small_recursive
7,small_recursive,doc_00000_chunk_007,doc_00000,audit.txt,txt,None,d:\Abhishek\Project\LangChain\cybersecurity-mu...,small_recursive
8,small_recursive,doc_00000_chunk_008,doc_00000,audit.txt,txt,None,d:\Abhishek\Project\LangChain\cybersecurity-mu...,small_recursive
9,small_recursive,doc_00000_chunk_009,doc_00000,audit.txt,txt,None,d:\Abhishek\Project\LangChain\cybersecurity-mu...,small_recursive


In [16]:
# ============================================================
# STEP 15 — Detect very small chunks
# ============================================================

small_chunk_rows = []

for strategy_name, chunked_docs in all_chunk_results.items():
    for doc in chunked_docs:
        char_count = len(doc.page_content)
        word_count = len(doc.page_content.split())

        if char_count < 150:
            small_chunk_rows.append({
                "strategy": strategy_name,
                "chunk_id": doc.metadata.get("chunk_id"),
                "file_name": doc.metadata.get("file_name"),
                "page_number": doc.metadata.get("page_number"),
                "char_count": char_count,
                "word_count": word_count,
                "text_preview": doc.page_content[:200].replace("\n", " ")
            })

small_chunks_df = pd.DataFrame(small_chunk_rows)

print("Number of very small chunks:", len(small_chunks_df))
display(small_chunks_df.head(20))

Number of very small chunks: 111


,strategy,chunk_id,file_name,page_number,char_count,word_count,text_preview
0,small_recursive,doc_00000_chunk_000,audit.txt,NaN,12,2,AUDIT TRAILS
1,small_recursive,doc_00000_chunk_027,audit.txt,NaN,147,22,"expose a variety of security violations, which..."
2,small_recursive,doc_00001_chunk_004,cui-ssp-template-final.docx,NaN,134,18,Include a detailed topology narrative and grap...
3,small_recursive,doc_00001_chunk_007,cui-ssp-template-final.docx,NaN,139,20,[Insert a system topology graphic. Provide a n...
4,small_recursive,doc_00001_chunk_011,cui-ssp-template-final.docx,NaN,72,12,. If the requirement is not applicable to the ...
5,small_recursive,doc_00004_chunk_001,NIST.CSWP.29.pdf,2.0,44,5,shared with NIST at cyberframework@nist.gov.
6,small_recursive,doc_00010_chunk_004,NIST.CSWP.29.pdf,8.0,85,9,cybersecurity incidents and enable appropriate...
7,small_recursive,doc_00010_chunk_006,NIST.CSWP.29.pdf,8.0,135,22,RECOVER Functions. GOVERN is in the center of ...
8,small_recursive,doc_00010_chunk_008,NIST.CSWP.29.pdf,8.0,95,13,the organization’s in-house data center to the...
9,small_recursive,doc_00011_chunk_000,NIST.CSWP.29.pdf,9.0,75,13,NIST CSWP 29 The NIST Cybersecurity Framework ...


In [17]:
# ============================================================
# STEP 16 — Compare chunking on one selected file
# ============================================================

selected_file_name = cleaned_documents[0].metadata.get("file_name") if cleaned_documents else None
print("Selected file for comparison:", selected_file_name)

for strategy_name, chunked_docs in all_chunk_results.items():
    matching_chunks = [
        doc for doc in chunked_docs
        if doc.metadata.get("file_name") == selected_file_name
    ]

    print("\n" + "=" * 120)
    print(f"STRATEGY: {strategy_name} | FILE: {selected_file_name} | NUM CHUNKS: {len(matching_chunks)}")
    print("=" * 120)

    for doc in matching_chunks[:3]:
        print("\nCHUNK ID:", doc.metadata.get("chunk_id"))
        print("PAGE:", doc.metadata.get("page_number"))
        print("CHARS:", len(doc.page_content))
        print(doc.page_content[:1000])
        print("\n" + "-" * 120)

Selected file for comparison: audit.txt

STRATEGY: small_recursive | FILE: audit.txt | NUM CHUNKS: 58

CHUNK ID: doc_00000_chunk_000
PAGE: None
CHARS: 12
AUDIT TRAILS

------------------------------------------------------------------------------------------------------------------------

CHUNK ID: doc_00000_chunk_001
PAGE: None
CHARS: 487
Audit trails maintain a record of system activity both by system and
application processes and by user activity of systems and applications. In
conjunction with appropriate tools and procedures, audit trails can assist
in detecting security violations, performance problems, and flaws in
applications. This bulletin focuses on audit trails as a technical control
and discusses the benefits and objectives of audit trails, the types of
audit trails, and some common implementation issues.

------------------------------------------------------------------------------------------------------------------------

CHUNK ID: doc_00000_chunk_002
PAGE: None
CHARS:

In [18]:
# ============================================================
# STEP 17 — Build inspection tables
# ============================================================

inspection_rows = []

for strategy_name, chunked_docs in all_chunk_results.items():
    for doc in chunked_docs:
        inspection_rows.append({
            "strategy": strategy_name,
            "chunk_id": doc.metadata.get("chunk_id"),
            "parent_document_id": doc.metadata.get("parent_document_id"),
            "file_name": doc.metadata.get("file_name"),
            "doc_type": doc.metadata.get("doc_type"),
            "page_number": doc.metadata.get("page_number"),
            "char_count": len(doc.page_content),
            "word_count": len(doc.page_content.split()),
            "text_preview": doc.page_content[:200].replace("\n", " ")
        })

inspection_df = pd.DataFrame(inspection_rows)
display(inspection_df.head(20))

,strategy,chunk_id,parent_document_id,file_name,doc_type,page_number,char_count,word_count,text_preview
0,small_recursive,doc_00000_chunk_000,doc_00000,audit.txt,txt,NaN,12,2,AUDIT TRAILS
1,small_recursive,doc_00000_chunk_001,doc_00000,audit.txt,txt,NaN,487,72,Audit trails maintain a record of system activ...
2,small_recursive,doc_00000_chunk_002,doc_00000,audit.txt,txt,NaN,466,73,An audit trail is a series of records of compu...
3,small_recursive,doc_00000_chunk_003,doc_00000,audit.txt,txt,NaN,413,71,Audit trails may be used as either a support f...
4,small_recursive,doc_00000_chunk_004,doc_00000,audit.txt,txt,NaN,261,33,BENEFITS AND OBJECTIVES Audit trails can provi...
5,small_recursive,doc_00000_chunk_005,doc_00000,audit.txt,txt,NaN,433,66,Individual Accountability Audit trails are a t...
6,small_recursive,doc_00000_chunk_006,doc_00000,audit.txt,txt,NaN,437,69,"For example, audit trails can be used in conce..."
7,small_recursive,doc_00000_chunk_007,doc_00000,audit.txt,txt,NaN,259,43,may be very resource-intensive.) Comparisons c...
8,small_recursive,doc_00000_chunk_008,doc_00000,audit.txt,txt,NaN,432,66,Audit trails work in concert with logical acce...
9,small_recursive,doc_00000_chunk_009,doc_00000,audit.txt,txt,NaN,438,71,to which they have legitimate access authoriza...


In [19]:
# ============================================================
# STEP 18 — Save outputs
# ============================================================

comparison_path = OUTPUT_DIR / "chunking_strategy_comparison.csv"
per_file_path = OUTPUT_DIR / "chunking_per_file_summary.csv"
inspection_path = OUTPUT_DIR / "chunking_inspection_preview.csv"
small_chunks_path = OUTPUT_DIR / "small_chunks_report.csv"

comparison_df.to_csv(comparison_path, index=False)
grouped_strategy_file_df.to_csv(per_file_path, index=False)
inspection_df.to_csv(inspection_path, index=False)
small_chunks_df.to_csv(small_chunks_path, index=False)

print("Saved:", comparison_path)
print("Saved:", per_file_path)
print("Saved:", inspection_path)
print("Saved:", small_chunks_path)

Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\chunked_docs\chunking_strategy_comparison.csv
Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\chunked_docs\chunking_per_file_summary.csv
Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\chunked_docs\chunking_inspection_preview.csv
Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\chunked_docs\small_chunks_report.csv


In [20]:
# ============================================================
# STEP 19 — Save JSON previews for each strategy
# ============================================================

for strategy_name, chunked_docs in all_chunk_results.items():
    export_data = []

    for doc in chunked_docs:
        export_data.append({
            "page_content": doc.page_content,
            "metadata": doc.metadata
        })

    export_path = OUTPUT_DIR / f"{strategy_name}_chunked_preview.json"

    with open(export_path, "w", encoding="utf-8") as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)

    print("Saved:", export_path)

Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\chunked_docs\small_recursive_chunked_preview.json
Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\chunked_docs\medium_recursive_chunked_preview.json
Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\chunked_docs\large_recursive_chunked_preview.json


In [21]:
# ============================================================
# STEP 20 — Recommend best chunking strategy
# ============================================================

print("Chunking Strategy Recommendation")
print("=" * 80)

print("""
Recommended default for this project:
- Strategy: medium_recursive
- Chunk size: 800
- Chunk overlap: 120

Why:
1. It usually balances context preservation and retrieval precision.
2. It is strong for policy, procedure, and compliance-style documents.
3. It avoids too many tiny fragments.
4. It still keeps chunks manageable for local embeddings and retrieval.
""")

Chunking Strategy Recommendation

Recommended default for this project:
- Strategy: medium_recursive
- Chunk size: 800
- Chunk overlap: 120

Why:
1. It usually balances context preservation and retrieval precision.
2. It is strong for policy, procedure, and compliance-style documents.
3. It avoids too many tiny fragments.
4. It still keeps chunks manageable for local embeddings and retrieval.



In [22]:
# ============================================================
# STEP 21 — Final conclusion
# ============================================================

print("Notebook 03 completed.")

print("\nStrategies tested:")
for strategy_name in chunking_strategies.keys():
    print("-", strategy_name)

print("\nRecommended starting strategy:")
print("- medium_recursive")
print("- chunk_size = 800")
print("- chunk_overlap = 120")

print("\nNext notebook: 04_embedding_and_vector_db_tests.ipynb")

Notebook 03 completed.

Strategies tested:
- small_recursive
- medium_recursive
- large_recursive

Recommended starting strategy:
- medium_recursive
- chunk_size = 800
- chunk_overlap = 120

Next notebook: 04_embedding_and_vector_db_tests.ipynb
